<h1 align = "Center">
Lotus Group Customer Behavior Analysis

<h2 align = "Center">
Exploratory Data Analysis — Customer Behaviour
</h1>

Now we understand the business.

Our objective is not simply to produce charts. We want to discover patterns that can help Lotus Group:

1. Understand how frequently customers purchase.
2. Identify high-value and low-value customers.
3. Understand customer spending behaviour.
4. Identify preferred product categories.
5. Investigate purchasing trends over time.
6. Identify behavioural patterns that may help predict future purchases and spending.

We want to approach these questions systematically.

In [1]:
import pandas as pd
from pathlib import Path

# Define path to folder
data_path = Path(r"../01_data/processed")

def load_csv (filename):
    return pd.read_csv(data_path/filename)

# Establishing the Overall Purchasing Picture

In [5]:
# load data for EDA
transaction = load_csv("transaction_data.csv")
orders = load_csv("orders_clean.csv")

# Overall purchasing overview

print("Unique Customers:", transaction["customer_id"].nunique())

print("Unique Orders:", transaction["order_id"].nunique())

print("Transaction Lines:", transaction["detail_id"].nunique())

print("Total Quantity Sold:", transaction["quantity"].sum())

print("Total Recorded Order Revenue:",
      orders["total_revenue"].sum())

print("Total Completed Order Revenue:",
      orders.loc[
          orders["order_status"] == "Completed",
          "total_revenue"
      ].sum())

print("Completed Orders:",
      (orders["order_status"] == "Completed").sum())

print("Returned Orders:",
      (orders["order_status"] == "Returned").sum())

print("Cancelled Orders:",
      (orders["order_status"] == "Cancelled").sum())

Unique Customers: 2945
Unique Orders: 12000
Transaction Lines: 25099
Total Quantity Sold: 43313
Total Recorded Order Revenue: 45350979.0
Total Completed Order Revenue: 39652714.0
Completed Orders: 10442
Returned Orders: 1063
Cancelled Orders: 495


These figures establish the baseline against which we'll interpret subsequent findings.

### Observations:

**Customer activity**: *2,945* of the *3,000* registered customers made at least one recorded purchase. The remaining *55* have no recorded orders.

**Transaction volume**: The *12,000* orders generated *25,099* product lines. This means an order can contain multiple products.

**Order outcomes**: Approximately *87%* of orders were completed, *8.9%* returned and *4.1%* cancelled.

**Revenue**: *EGP 39.65 million* is the revenue associated with completed orders. We'll use this as our initial revenue measure for customer spending analysis.

**Caution**: these are three-year aggregate figures. They tell us the scale of the business, but not whether individual customers are becoming more or less valuable over time.

## Understanding Customer Purchasing Frequency

This is our first proper customer-level investigation.

We want to establish:
- How many orders does a typical customer place?
- What proportion of customers are one-time buyers?
- How many customers make repeat purchases?
- Are purchases concentrated among a small group of customers?

Here, only completed orders count as purchases.

In [6]:
# Retain completed orders only
completed_orders = orders[
    orders["order_status"] == "Completed"
].copy()

# Calculate orders per customer
customer_frequency = (
    completed_orders
    .groupby("customer_id")
    .agg(
        total_orders=("order_id", "nunique")
    )
    .reset_index()
)

# Summarize purchasing frequency
print("Customer Purchasing Frequency")
print(customer_frequency["total_orders"].describe())

print("\nCustomers by Number of Orders")
print(customer_frequency["total_orders"].value_counts().sort_index())

print("\nOne-Time Customers:",
      (customer_frequency["total_orders"] == 1).sum())

print("Repeat Customers:",
      (customer_frequency["total_orders"] > 1).sum())

Customer Purchasing Frequency
count    2905.000000
mean        3.594492
std         1.736648
min         1.000000
25%         2.000000
50%         3.000000
75%         5.000000
max        11.000000
Name: total_orders, dtype: float64

Customers by Number of Orders
total_orders
1     304
2     543
3     654
4     601
5     417
6     201
7     117
8      46
9      14
10      6
11      2
Name: count, dtype: int64

One-Time Customers: 304
Repeat Customers: 2601


**What we're learning**

This analysis helps us distinguish between one-time and repeat customers.

This distinction matters because our eventual classification model will attempt to predict which customers are likely to make another qualifying purchase within the next 90 days.

**Key findings**

1. Strong repeat-purchase behaviour

    - Approximately 89.5% of customers with completed purchases placed more than one order.

        - This suggests that repeat purchasing is an important component of Lotus Group's business.

2. Most customers purchase relatively frequently

    - The median customer placed three orders during the observation period. Half of the customers placed three or fewer orders, while the other half placed three or more.

3. A smaller group purchases very frequently

    - Only two customers placed 11 orders, while eight or more orders were relatively uncommon.

**An important distinction**

We have:
- 3,000 registered customers.
- 2,945 customers with at least one recorded order, regardless of status.
- 2,905 customers with at least one completed order.

The difference matters. Some customers have only returned or cancelled orders, while 55 have no recorded orders at all.

We'll preserve these distinctions when constructing our modelling population.

## Investigating Customer Spending Behavior

Purchasing frequency tells us how often customers buy. It doesn't tell us how valuable those customers are, though.

Note that a customer who makes two EGP 10,000 purchases may be more valuable than someone who makes five EGP 500 purchases.

We therefore need to calculate total completed-order spending per customer.

In [ ]:
# Calculate completed spending per customer

customer_spending = (
    completed_orders
    .groupby("customer_id")
    .agg(
        total_spending=("total_revenue", "sum"),
        total_orders=("order_id", "nunique"),
        average_order_value=("total_revenue", "mean")
    )
    .reset_index()
)

print("Customer Spending Summary")

print(customer_spending[
    ["total_spending", "total_orders", "average_order_value"]
].describe().round(2))

CUSTOMER SPENDING SUMMARY
       total_spending  total_orders  average_order_value
count         2905.00       2905.00              2905.00
mean         13649.82          3.59              3757.83
std          17563.91          1.74              4830.70
min             67.50          1.00                67.50
25%           2439.00          2.00               856.25
50%           6127.50          3.00              1693.75
75%          18384.00          5.00              4787.70
max         123695.00         11.00             55747.00


What do these three measures tell us?

|Measure	|What it tells us|
|---|---|
|Total spending	|The total completed-order value contributed by a customer.|
|Total orders	|How frequently the customer purchases.|
|Average order value	|The average value of each completed order.|


These measures should help us distinguish high-value customers from frequent but low-spending customers.

**Finding 1: Spending is highly dispersed**

* The average customer spending is EGP 13,649.82, whereas the median is only EGP 6,127.50.

* This difference suggests that some customers spend considerably more than the majority, pulling the average upwards.

* The standard deviation of EGP 17,563.91 reinforces this observation.

**Finding 2: Customer value varies substantially**

* The highest-spending customer contributed EGP 123,695, compared with EGP 67.50 for the lowest.

* This suggests that treating all customers as equally valuable would obscure important differences in purchasing behaviour.

**Finding 3: Order value also varies considerably**

* The average order value is EGP 3,757.83, but the median customer-level average order value is EGP 1,693.75.

* This indicates that a number of customers make relatively large purchases.

**Important distinction**: The EGP 3,757.83 figure is the average order value across all completed orders, while EGP 1,693.75 is the median of customers' individual average order values. They measure different things.